In [2]:
# ---------- Ultra-Minimal (multi-char: JP via JapanDict; JD-miss => fugashi+kakasi/GCT; KO via local; VN only in Definition; Nôm table with VI TTS on Quốc Ngữ) ----------
from IPython.display import display, clear_output, HTML, Markdown
import ipywidgets as widgets
from pathlib import Path
import pandas as pd
import re, unicodedata, base64, hashlib, subprocess, csv, itertools, warnings, html as _html
from urllib.parse import quote_plus
import urllib.parse
from bs4 import BeautifulSoup
import requests
from io import StringIO
from docx import Document  # for Sinitic paper DOCX

# Hanja/Hangul + romanization
from hanja import translate as hanja_to
from hangul_romanize import Transliter
from hangul_romanize.rule import academic as _ko_rule
_KO_TRANSLITER = Transliter(_ko_rule)

# Romaji -> kana helper (separate file)
from kana_romaji_utils import romaji_to_hiragana

# ===== Paths (portable: env-overridable, resolved from the repo root) =====
import os

def _resolve_root() -> Path:
    env = os.environ.get("HANZIHUB_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if (cand / "hanzihub").is_dir() or (cand / "Unihan_Readings.txt").exists():
            return cand
    return here

ROOT = _resolve_root()

# Prefer the new provenanced build output; fall back to the legacy tagged CSV.
_NEW_CSV     = ROOT / "data" / "out" / "combined_readings.csv"
_LEGACY_CSV  = ROOT / "combined_dialects_imputed_tagged_excl_jp_kr.csv"
COMBINED_CSV = str(_NEW_CSV if _NEW_CSV.exists() else _LEGACY_CSV)

IPA_FILE     = str(ROOT / "ipa_full_library_complete.txt")
CACHE_DIR    = str(ROOT / "cache")
EXCLUDE_COLS = {"fanqie_upper", "fanqie_lower"}

# ===== Guides (README) =====
GUIDES_DIR = ROOT / "Guides"
GUIDE_FILES = {
    "IPA Guide": "IPA_guide.md",
    "Tone Contour Guide": "tone_contour_guide.md",
    "Baxter–Sagart OC/MC Guide": "baxtersagart_guide.md",
}

# ===== Voices for TTS =====
VOICE_JA  = ("ja-JP",  "ja-JP-Wavenet-B")
VOICE_KO  = ("ko-KR",  "ko-KR-Wavenet-A")
VOICE_ZH  = ("cmn-CN", "cmn-CN-Wavenet-A")
VOICE_YUE = ("yue-HK", "yue-HK-Standard-A")
VOICE_VI  = ("vi-VN",  "vi-VN-Wavenet-A")

ENABLE_TOPOLECT_FALLBACK_CHAR_TTS = False
TOPOLECT_FALLBACK_VOICE = VOICE_YUE
TOPOLECT_LABELS = {"Teochew", "Hokkien", "Hakka", "Wu", "Gan", "Xiang", "Jin"}

# ===== Google Cloud Translation (v3) =====
PROJECT_ID = "zach-ipa-tts-2025"
PARENT = f"projects/{PROJECT_ID}/locations/global"

# Credentials come from Application Default Credentials. Point
#   GOOGLE_APPLICATION_CREDENTIALS=<service-account key path>
# at the key — never hard-code or commit it. For backward-compat on the
# original machine we fall back to the in-repo key only if that env var is
# unset and the file still exists (rotate that key — see README).
try:
    from google.cloud import translate_v3 as translate
    if os.environ.get("GOOGLE_APPLICATION_CREDENTIALS"):
        _gt_client = translate.TranslationServiceClient()
    else:
        _legacy_key = ROOT / "TTS" / "tts-robot-key.json"
        _gt_client = (
            translate.TranslationServiceClient.from_service_account_file(str(_legacy_key))
            if _legacy_key.exists()
            else translate.TranslationServiceClient()
        )
except Exception:
    _gt_client = None
_GT_READY = _gt_client is not None

def _maybe_diag():
    if not _GT_READY:
        display(HTML("<i style='color:#888'>Cloud Translation client not initialized — check API enable/billing/credentials.</i>"))

def gtrans_translate_text(text: str, target: str, source: str = None) -> str:
    """Translate with optional explicit source language."""
    if not _gt_client or not text:
        return ""
    try:
        req = {"parent": PARENT, "contents": [text], "target_language_code": target}
        if source:
            req["source_language_code"] = source
        resp = _gt_client.translate_text(request=req)
        if resp.translations:
            return resp.translations[0].translated_text or ""
    except Exception:
        return ""
    return ""

def gtrans_romanize(text: str, source_lang: str) -> str:
    if not _gt_client or not text:
        return ""
    try:
        resp = _gt_client.romanize_text(
            request={"parent": PARENT, "contents": [text], "source_language_code": source_lang}
        )
        if resp.romanizations:
            return resp.romanizations[0].romanized_text or ""
    except Exception:
        return ""
    return ""

def _rom_zh(s: str) -> str:
    return gtrans_romanize(s, "zh") or ""

def _rom_ko(s: str) -> str:
    try:
        return _KO_TRANSLITER.translit(s) if s else ""
    except Exception:
        return ""

# ===== Japanese shinjitai mapping (your JPVariants.txt) =====
def load_jpvariant_dict(txt_path):
    mapping = {}
    try:
        with open(txt_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    trad, simp = parts[0], parts[1]
                    if len(simp) == 1:
                        mapping[trad] = simp
    except Exception:
        mapping = {}
    return mapping

try:
    import opencc
    JPVAR_PATH = os.path.join(os.path.dirname(opencc.__file__), "dictionary", "JPVariants.txt")
except Exception:
    JPVAR_PATH = ""
JPVAR_MAP = load_jpvariant_dict(JPVAR_PATH)
print(f"Loaded {len(JPVAR_MAP)} variant mappings (jp).")

def to_shinjitai_per_char(s: str) -> str:
    return "".join(JPVAR_MAP.get(ch, ch) for ch in (s or ""))

# ===== JapanDict scraping (per-kanji: TTS kana/romaji) =====
_JD_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}
# meta: "<kana> - <romaji> - ..."
_RE_JD_META = re.compile(r"([ぁ-ゖァ-ヺー]+)\s*-\s*([A-Za-z\- ]+)\s*-\s")

def ja_romaji_and_kana(kanji_text: str):
    """
    Prefer JapanDict (kana + romaji) via meta description.
    Fallback:
      - kana via fugashi (MeCab) if available
      - romaji via pykakasi over kana
    """
    if not kanji_text:
        return "", ""
    jp_text = to_shinjitai_per_char(kanji_text)

    # 1) Try JapanDict
    try:
        url = "https://www.japandict.com/" + urllib.parse.quote(jp_text)
        r = requests.get(url, headers=_JD_HEADERS, timeout=12)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        meta = soup.find("meta", attrs={"name": "description"})
        if meta:
            m = _RE_JD_META.search(meta.get("content", ""))
            if m:
                kana = m.group(1).strip()
                romaji = m.group(2).strip()
                # katakana → hiragana
                kana = "".join(
                    chr(ord(ch) - 0x60) if "ァ" <= ch <= "ヶ" else ch for ch in kana
                )
                return romaji, kana
    except Exception:
        pass

    # 2) Fallback kana via fugashi
    kana = ""
    try:
        from fugashi import Tagger as _ja_Tagger

        tagger = _ja_Tagger()
        kana = "".join([(w.feature.kana or w.surface) for w in tagger(jp_text)])
        kana = "".join(
            chr(ord(ch) - 0x60) if "ァ" <= ch <= "ヶ" else ch for ch in kana
        )
    except Exception:
        kana = ""

    # 3) Romaji via pykakasi over kana
    romaji = ""
    if kana:
        try:
            import pykakasi as _pykakasi

            kks = _pykakasi.kakasi()
            romaji = "".join([x["hepburn"] for x in kks.convert(kana)]).strip()
        except Exception:
            romaji = ""

    if not kana and not romaji:
        return "", ""

    return romaji.strip(), kana.strip()

# ===== JP TTS safety helpers (fix は→“wa” particle heuristic) =====
def _kana_hira_to_kata(s: str) -> str:
    # hiragana → katakana; leave others as-is
    return "".join(
        chr(ord(ch) + 0x60) if "ぁ" <= ch <= "ゖ" else ch for ch in (s or "")
    )

def _prep_ja_tts(kana: str, fallback_kanji: str) -> str:
    # strip spaces to avoid token boundaries; prefer katakana so は isn't treated as particle
    k = re.sub(r"\s+", "", (kana or ""))
    return _kana_hira_to_kata(k) if k else fallback_kanji

# ===== Vietnamese Nôm (Nom Foundation) =====
warnings.filterwarnings("ignore", category=FutureWarning, module="pandas.io.html")

def fetch_nom_df_from_hanzi(hanzi: str):
    """Hanzi → Nôm (input_type=rqn_or_hn), used in the main Hanzi lookup."""
    hanzi = (hanzi or "").strip()
    if not hanzi:
        return None, None
    url = (
        "https://nomfoundation.org/nom-tools/Nom-Lookup-Tool/Nom-Lookup-Tool"
        f"?input_type=rqn_or_hn&inputText={quote_plus(hanzi)}&uiLang=en&GO=GO"
    )
    resp = requests.get(url, timeout=12)
    resp.encoding = "utf-8"
    soup = BeautifulSoup(resp.text, "html.parser")
    dfs = []
    for table in soup.find_all("table"):
        try:
            df = pd.read_html(StringIO(str(table)))[0]
            dfs.append(df)
        except Exception:
            pass
    df = next(
        (
            d
            for d in dfs
            if any(
                c in ("Quốc Ngữ", "Quốc ngữ", "Qu\u1ed1c Ng\u1eef") for c in d.columns
            )
        ),
        None,
    )
    return df, url

def fetch_nom_dict_table_from_vi_or_en(text: str):
    """Vietnamese Quốc Ngữ / Chinese text → main Nôm dictionary table (Quốc Ngữ / Hán-Nôm / English)."""
    text = (text or "").strip()
    if not text:
        return None, None

    url = (
        "https://www.nomfoundation.org/nom-tools/Nom-Lookup-Tool/Nom-Lookup-Tool"
        f"?input_type=rqn_or_hn&inputText={quote_plus(text)}&uiLang=en&GO=GO"
    )
    resp = requests.get(url, timeout=12)
    resp.encoding = "utf-8"
    soup = BeautifulSoup(resp.text, "html.parser")

    good_df = None
    for table in soup.find_all("table"):
        try:
            df = pd.read_html(StringIO(str(table)))[0]
        except Exception:
            continue

        cols = [str(c) for c in df.columns]
        has_qn = any(
            "Quốc Ngữ" in c or "Qu\u1ed1c Ng\u1eef" in c for c in cols
        )
        has_hn = any(
            "Hán-Nôm" in c or "Hán Nôm" in c or "Hán-nôm" in c for c in cols
        )
        has_eng = any("English" in c for c in cols)

        if has_qn and has_hn and has_eng:
            good_df = df
            break

    return good_df, url

# ===== TTS (ZH / YUE / JA / KO / VI) =====
try:
    from google.cloud import texttospeech as gcp_tts
except Exception:
    gcp_tts = None

def _tts_cache_path(token, lang, voice):
    key = hashlib.sha1(f"{token}|{lang}|{voice}".encode("utf-8")).hexdigest()
    p = Path(CACHE_DIR) / "tts_cache" / f"{key}.mp3"
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

def _synth_gcp(text, lang, voice):
    if gcp_tts is None:
        raise RuntimeError("google.cloud.texttospeech not available")
    client = gcp_tts.TextToSpeechClient()
    sinput = gcp_tts.SynthesisInput(text=text)
    try:
        vsel = gcp_tts.VoiceSelectionParams(language_code=lang, name=voice)
        aconf = gcp_tts.AudioConfig(audio_encoding=gcp_tts.AudioEncoding.MP3)
        resp = client.synthesize_speech(
            input=sinput, voice=vsel, audio_config=aconf
        )
        return resp.audio_content
    except Exception:
        vsel = gcp_tts.VoiceSelectionParams(language_code=lang)
        aconf = gcp_tts.AudioConfig(audio_encoding=gcp_tts.AudioEncoding.MP3)
        resp = client.synthesize_speech(
            input=sinput, voice=vsel, audio_config=aconf
        )
        return resp.audio_content

def _synth_espeak(text, lang_code, voice_name):
    wav = Path.cwd() / ".tmp_espeak.wav"
    subprocess.run(
        ["espeak-ng", "-w", str(wav), text], check=True, capture_output=True
    )
    try:
        mp3 = Path.cwd() / ".tmp_espeak.mp3"
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-i", str(wav), str(mp3)],
            check=True,
            capture_output=True,
        )
        data = mp3.read_bytes()
        mp3.unlink(missing_ok=True)
    except Exception:
        data = wav.read_bytes()
    wav.unlink(missing_ok=True)
    return data

def tts_bytes(text, lang, voice):
    p = _tts_cache_path(text, lang, voice or "")
    if p.exists():
        return p.read_bytes()
    last = None
    try:
        data = _synth_gcp(text, lang, voice)
        p.write_bytes(data)
        return data
    except Exception as e:
        last = e
    try:
        data = _synth_espeak(text, lang, voice)
        p.write_bytes(data)
        return data
    except Exception as e2:
        raise RuntimeError(f"TTS failed (GCP: {last}); espeak: {e2}")

def play_audio(mp3_bytes):
    b64 = base64.b64encode(mp3_bytes).decode("utf-8")
    display(
        HTML(
            '<audio autoplay controls style="width:240px">'
            f'<source src="data:audio/mp3;base64,{b64}" type="audio/mp3"></audio>'
        )
    )

def _make_button_and_play(label_text, lang, voice, width=180, speak=None):
    btn = widgets.Button(
        description=f"▶ {label_text}",
        layout=widgets.Layout(width=f"{width}px"),
    )
    box = widgets.Output()

    def _go(_b):
        with box:
            clear_output()
            try:
                say = speak if (speak is not None and str(speak).strip()) else label_text
                data = tts_bytes(say, lang, voice)
                play_audio(data)
            except Exception as e:
                print("TTS error:", e)

    btn.on_click(_go)
    display(widgets.HBox([btn, box]))

# ===== Render Nôm table with VI TTS on Quốc Ngữ =====
def render_nom_table_with_vi_tts(df):
    if df is None or df.empty:
        display(HTML("<i style='color:#888'>No Nôm dictionary results.</i>"))
        return
    qn_col = next(
        (
            c
            for c in df.columns
            if str(c) in ("Quốc Ngữ", "Quốc ngữ", "Qu\u1ed1c Ng\u1eef")
        ),
        None,
    )
    rows_html = []
    uid = 0
    for _, row in df.iterrows():
        cells = []
        for col in df.columns:
            val = str(row[col]) if pd.notna(row[col]) else ""
            if qn_col is not None and col == qn_col and val.strip():
                uid += 1
                try:
                    mp3 = tts_bytes(val, *VOICE_VI)
                    b64 = base64.b64encode(mp3).decode("utf-8")
                    aud_id = f"vi_aud_{uid}"
                    cell_html = (
                        f'<audio id="{aud_id}" preload="auto" style="display:none">'
                        f'<source src="data:audio/mp3;base64,{b64}" type="audio/mp3"></audio>'
                        f'<a href="#" onclick="document.getElementById(\'{aud_id}\').play();return false;" '
                        f'style="text-decoration:none;">{_html.escape(val)}</a>'
                    )
                except Exception:
                    cell_html = _html.escape(val)
                cells.append(f"<td>{cell_html}</td>")
            else:
                cells.append(f"<td>{_html.escape(val)}</td>")
        rows_html.append("<tr>" + "".join(cells) + "</tr>")
    thead = (
        "<tr>"
        + "".join(f"<th>{_html.escape(str(c))}</th>" for c in df.columns)
        + "</tr>"
    )
    html = (
        '<div style="max-height:420px; overflow:auto; border:1px solid #444; border-radius:8px; padding:6px;">'
        '<table style="width:100%; font-size:90%; border-collapse:collapse;">'
        f"{thead}"
        + "".join(rows_html)
        + "</table></div>"
    )
    display(HTML(html))

def show_nom_table_from_hanzi(hanzi: str):
    df, _ = fetch_nom_df_from_hanzi(hanzi)
    render_nom_table_with_vi_tts(df)

# ===== koreanhanja.app helpers (Hanzi + English, unified) =====

def fetch_hanja_rows_from_hanzi(hanzi: str):
    """
    Hanzi (possibly multi-char) -> per-character rows from koreanhanja.app/<hanzi>,
    using the same table-based parsing as your old working code.

    Returns a list of (hanja, hangul, meaning) tuples that
    render_koreanhanja_table(...) can consume.
    """
    hanzi = (hanzi or "").strip()
    if not hanzi:
        return []

    url = f"https://koreanhanja.app/{quote_plus(hanzi)}"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    rows = []
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) >= 3:
            hanja_cell  = tds[0].get_text(" ", strip=True)
            hangul_cell = tds[1].get_text(" ", strip=True)
            gloss_cell  = tds[2].get_text(" ", strip=True)
            rows.append((hanja_cell, hangul_cell, gloss_cell))
        elif len(tds) >= 2:
            hanja_cell  = tds[0].get_text(" ", strip=True)
            gloss_cell  = tds[1].get_text(" ", strip=True)
            rows.append((hanja_cell, "", gloss_cell))

    return rows


def fetch_hanja_rows_from_english(eng: str):
    """
    English → koreanhanja.app search page (table layout),
    used by the reverse (English) pipeline.

    Returns (rows, url) where rows is a list of (hanja, hangul, meaning).
    """
    eng = (eng or "").strip()
    if not eng:
        return [], None

    url = f"https://koreanhanja.app/{quote_plus(eng)}"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    rows = []
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) >= 3:
            hanja  = tds[0].get_text(" ", strip=True)
            hangul = tds[1].get_text(" ", strip=True)
            gloss  = tds[2].get_text(" ", strip=True)
            rows.append((hanja, hangul, gloss))

    # optional filter so it roughly matches the English gloss
    q = eng.lower()
    rows = [r for r in rows if q in (r[2] or "").lower()]
    return rows, url


def render_koreanhanja_table(rows, source_url=None):
    """
    Render koreanhanja.app rows as a table with clickable Hangul that plays KO TTS.
    Used by BOTH:
      * Hanzi pipeline (show_hanja_rows_from_hanzi)
      * English reverse pipeline (on_reverse_lookup)
    """
    if not rows:
        display(HTML("<i style='color:#888'>No Korean hanja results for this query.</i>"))
        if source_url:
            display(HTML(f'<a href="{source_url}" target="_blank">View on koreanhanja.app</a>'))
        return

    head = "<tr><th>Hanja</th><th>Hangul</th><th>Meaning</th></tr>"
    body_rows = []
    uid = 0

    for h, hg, m in rows:
        hg = hg or ""

        if hg.strip():
            uid += 1
            try:
                mp3 = tts_bytes(hg, *VOICE_KO)
                b64 = base64.b64encode(mp3).decode("utf-8")
                aud_id = f"ko_aud_{uid}"
                hangul_html = (
                    f'<audio id="{aud_id}" preload="auto" style="display:none">'
                    f'<source src="data:audio/mp3;base64,{b64}" type="audio/mp3"></audio>'
                    f'<a href="#" onclick="document.getElementById(\'{aud_id}\').play();'
                    f"return false;\" style=\"text-decoration:none;\">"
                    f"{_html.escape(hg)}</a>"
                )
            except Exception:
                hangul_html = _html.escape(hg)
        else:
            hangul_html = ""

        row_html = (
            "<tr>"
            f"<td>{_html.escape(h or '')}</td>"
            f"<td>{hangul_html}</td>"
            f"<td>{_html.escape(m or '')}</td>"
            "</tr>"
        )
        body_rows.append(row_html)

    html = f"""
    <div style="max-height:420px; overflow:auto; border:1px solid #444;
                border-radius:8px; padding:6px;">
      <table style="width:100%; font-size:90%; border-collapse:collapse;">
        {head}
        {''.join(body_rows)}
      </table>
    </div>
    """
    display(HTML(html))
    if source_url:
        display(HTML(f'<a href="{source_url}" target="_blank">View on koreanhanja.app</a>'))


def show_hanja_rows_from_hanzi(hanzi: str):
    """
    Wrapper used in the Chinese (multi-character Hanzi) pipeline.
    """
    hanzi = (hanzi or "").strip()
    rows = fetch_hanja_rows_from_hanzi(hanzi)
    render_koreanhanja_table(rows, f"https://koreanhanja.app/{quote_plus(hanzi)}")


def show_hanja_rows_from_english_gloss(en_gloss: str):
    """
    Optional helper if you ever want to go:
      Chinese → English gloss → koreanhanja.app

    Not used by default in your current UI, but safe to keep.
    """
    rows, url = fetch_hanja_rows_from_english(en_gloss)
    render_koreanhanja_table(rows, url)

# ===== JapanDict (English search, with JA TTS on headword via Reading) =====
_JP_CHAR_RE = re.compile(r"[ぁ-ゖァ-ヺ一-龯]")
_JD_BASE_URL = "https://www.japandict.com/"

def _fetch_japandict_entries_page(
    session: requests.Session, query: str, lang: str = "eng", page: int = 1
):
    """
    Fetch ONE search-results page from JapanDict.
    Returns list of dicts (including the entry href so we can detect duplicates).
    """
    if page <= 1:
        url = f"{_JD_BASE_URL}?s={quote_plus(query)}&lang={lang}"
    else:
        url = f"{_JD_BASE_URL}?s={quote_plus(query)}&page={page}&lang={lang}"

    r = session.get(url, headers=_JD_HEADERS, timeout=12)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    results = []

    for a in soup.select("a.list-group-item.list-group-item-action"):
        href = a.get("href")

        # Japanese headword
        headword_el = a.select_one("span.xlarge.text-normal")
        headword = headword_el.get_text(strip=True) if headword_el else None

        # Reading (romaji)
        reading_el = a.select_one("i.text-muted.xsmall")
        reading = reading_el.get_text(strip=True) if reading_el else None

        # Meanings
        meaning_els = a.select("ul.list-unstyled li")
        meanings = [m.get_text(strip=True) for m in meaning_els]

        results.append(
            {
                "href": href,
                "headword": headword,
                "reading": reading,
                "meanings": meanings,
            }
        )

    return results

def fetch_japandict_from_english(eng: str, max_pages: int = 50):
    """
    Fetch all JapanDict results for an English query.
    Returns (entries, first_page_url).
    entries is a list of dicts: {headword, reading, meanings}.
    """
    query = (eng or "").strip()
    if not query:
        return [], None

    all_results = []
    seen_hrefs = set()
    first_url = f"{_JD_BASE_URL}?s={quote_plus(query)}&lang=eng"

    with requests.Session() as session:
        for page in range(1, max_pages + 1):
            page_results = _fetch_japandict_entries_page(
                session, query, lang="eng", page=page
            )

            if not page_results:
                break

            new_any = False
            for entry in page_results:
                h = entry.get("href")
                if h not in seen_hrefs:
                    seen_hrefs.add(h)
                    all_results.append(entry)
                    new_any = True

            if not new_any:
                # this page was all duplicates → we are past the last page
                break

    return all_results, first_url

def render_japandict_entries(entries, source_url=None):
    """
    Render JapanDict search results as a table with clickable Japanese audio.

    TTS pipeline:
      - Take the 'reading' field (romaji),
      - Convert romaji -> hiragana via romaji_to_hiragana,
      - Convert hiragana -> katakana,
      - Feed katakana into JA TTS.

    This way audio lines up with the romaji reading and the headword.
    """
    if not entries:
        display(
            HTML(
                "<i style='color:#888'>No JapanDict entries parsed for this query.</i>"
            )
        )
        if source_url:
            display(
                HTML(f'<a href="{source_url}" target="_blank">More on JapanDict</a>')
            )
        return

    head = "<tr><th>Japanese</th><th>Reading</th><th>Meanings</th></tr>"
    rows = []
    uid = 0

    for entry in entries:
        # ---- Normalise structure ----
        if isinstance(entry, dict):
            jp = entry.get("headword") or entry.get("Japanese") or ""
            reading = entry.get("reading") or ""
            meanings = entry.get("meanings") or ""
            if isinstance(meanings, (list, tuple)):
                meaning_text = "; ".join(str(m) for m in meanings if m)
            else:
                meaning_text = str(meanings)
        else:
            # tuple / list fallback: (jp, gloss)
            jp = entry[0] if len(entry) > 0 else ""
            reading = ""
            meaning_text = entry[1] if len(entry) > 1 else ""

        jp = jp or ""
        reading = reading or ""

        # ---- Build TTS text from romaji reading ----
        tts_text = jp
        if reading.strip():
            hira = romaji_to_hiragana(reading)
            if hira:
                tts_text = _kana_hira_to_kata(hira)

        jp_cell = _html.escape(jp)
        if tts_text.strip():
            try:
                audio_bytes = tts_bytes(tts_text, *VOICE_JA)
                b64 = base64.b64encode(audio_bytes).decode("utf-8")
                uid += 1
                aud_id = f"ja_dict_{uid}"
                jp_cell = (
                    f'<audio id="{aud_id}" preload="auto" style="display:none">'
                    f'<source src="data:audio/mp3;base64,{b64}" type="audio/mp3"></audio>'
                    f'<a href="#" onclick="document.getElementById(\'{aud_id}\').play();'
                    f"return false;\" style=\"text-decoration:none;\">"
                    f"{_html.escape(jp)}</a>"
                )
            except Exception:
                jp_cell = _html.escape(jp)

        rows.append(
            "<tr>"
            f"<td>{jp_cell}</td>"
            f"<td>{_html.escape(reading)}</td>"
            f"<td>{_html.escape(meaning_text)}</td>"
            "</tr>"
        )

    html = f"""
    <div style="max-height:420px; overflow:auto; border:1px solid #444;
                border-radius:8px; padding:6px;">
      <table style="width:100%; font-size:90%; border-collapse:collapse;">
        {head}
        {''.join(rows)}
      </table>
    </div>
    """
    display(HTML(html))
    if source_url:
        display(
            HTML(f'<a href="{source_url}" target="_blank">More on JapanDict</a>')
        )

# ===== Small helper for toggle buttons =====
def make_toggle_list_button(title: str, render_fn, arg_text: str, width=200):
    btn = widgets.ToggleButton(
        value=False,
        description=title,
        icon="search",
        layout=widgets.Layout(width=f"{width}px"),
    )
    out = widgets.Output()

    def _on_change(change):
        if change["name"] != "value":
            return
        with out:
            clear_output()
            if change["new"]:
                try:
                    render_fn(arg_text)
                except Exception as e:
                    display(HTML(f"<i style='color:#c66'>Error: {e}</i>"))

    btn.observe(_on_change, names="value")
    return btn, out

# ===== Misc helpers =====
def _norm(s: str) -> str:
    return re.sub(r"\s+", " ", str(s or "").strip()).lower()

def _inline_tokens(val: str) -> str:
    toks = [
        t
        for t in re.split(r"[\/\|,;]+\s*|\s+", (val or "").strip())
        if t
    ]
    return " ".join(toks)

def load_phonetic_symbols(csv_path):
    out = {}
    if not csv_path or not Path(csv_path).exists():
        return out
    with open(csv_path, "r", encoding="utf-8-sig", newline="") as f:
        sample = f.read(8192)
        f.seek(0)
        try:
            delim = csv.Sniffer().sniff(sample).delimiter
        except Exception:
            delim = "\t" if "\t" in sample else ","
        reader = csv.reader(f, delimiter=delim)
        first = next(reader, None)
        if first is None:
            return out
        header = [c.strip().lower() for c in first]
        header_like = any(
            any(k in c for k in ("ipa_symbol", "ipa", "symbol", "phonetic", "phoneme"))
            for c in header
        )
        if header_like:
            sym_idx = next(
                (
                    i
                    for i, c in enumerate(header)
                    if any(
                        k in c
                        for k in ("ipa_symbol", "ipa", "symbol", "phonetic", "phoneme")
                    )
                ),
                0,
            )
            desc_idx = next(
                (
                    i
                    for i, c in enumerate(header)
                    if any(
                        k in c for k in ("description", "desc", "meaning", "example")
                    )
                ),
                None,
            )
            rows_iter = reader
        else:
            sym_idx = 0
            desc_idx = 1 if len(first) > 1 else None
            rows_iter = itertools.chain([first], reader)
        for row in rows_iter:
            if not row or len(row) <= sym_idx:
                continue
            raw_sym = (row[sym_idx] or "").strip()
            if not raw_sym:
                continue
            if desc_idx is not None and len(row) > desc_idx:
                desc = " ".join(
                    (row[i] or "").strip()
                    for i in range(desc_idx, len(row))
                    if (row[i] or "").strip()
                )
            else:
                desc = (
                    " ".join(
                        (x or "").strip()
                        for x in row[1:]
                        if (x or "").strip()
                    )
                    if len(row) > 1
                    else ""
                )
            key = unicodedata.normalize("NFC", raw_sym)
            desc = unicodedata.normalize("NFC", desc)
            out[key] = (desc, "")
    return out

def remove_cjk_chars(text):
    if not text:
        return ""
    return re.sub(r"[\u3400-\u9FFF\U00020000-\U0002FFFF]+", "", str(text))

def show_ipa_legend_lines(pron_strs, phonetic_symbols, debug=False):
    if not phonetic_symbols:
        return []
    pron_strs_n = [
        unicodedata.normalize("NFC", s) for s in (pron_strs or []) if s
    ]
    combined = " ".join(remove_cjk_chars(s) for s in pron_strs_n)
    combined = unicodedata.normalize("NFC", combined)
    TIE_CANON = "\u0361"
    for v in {"\u0361", "\u203F", "\u035C", "\u035D", "\u02D5", "͡"}:
        combined = combined.replace(v, TIE_CANON)
    if not combined.strip():
        return []
    keys = sorted(
        [
            unicodedata.normalize("NFC", k).replace("\u0361", TIE_CANON)
            for k in phonetic_symbols.keys()
            if k
        ],
        key=len,
        reverse=True,
    )
    if not keys:
        return []
    pattern = "|".join(re.escape(k) for k in keys)
    seen, seen_set = [], set()
    for m in re.finditer(pattern, combined):
        sym = m.group(0)
        if sym not in seen_set and sym in phonetic_symbols:
            seen.append(sym)
            seen_set.add(sym)
    return [f"  {sym}: {phonetic_symbols[sym][0]}" for sym in seen]

PREFERRED_ORDER = [
    "Definition",
    "Mandarin", "Cantonese",
    "Hokkien", "Teochew", "Eastern Min (Fuzhou)",
    "Northern Min (Jian'ou)", "Hainanese (Haikou)",
    "Hakka", "Wu", "Hui (Shexian)", "Gan", "Xiang", "Jin",
    "Middle Chinese", "Middle Chinese (Baxter)", "Middle Chinese (position)",
    "Tang", "Fanqie", "fanqie_upper", "fanqie_lower",
    "Old Chinese", "Old Chinese (Zhengzhang)",
    "Vietnamese", "Zhuang",
    "Japanese", "Japanese On", "Japanese Kun",
    "Korean (Hangul)", "Korean (romanized)", "Korean (Yale)",
]

def load_from_combined_csv(csv_path):
    df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
    cols = list(df.columns)
    ch_col = next(
        (c for c in cols if _norm(c) in {"char", "character", "zi", "漢字", "字"}),
        cols[0],
    )
    _EXCL_N = {re.sub(r"\s+", " ", c.lower()) for c in EXCLUDE_COLS}
    dialect_cols = [
        c
        for c in cols
        if c != ch_col and re.sub(r"\s+", " ", c.lower()) not in _EXCL_N
    ]
    data = {}
    for _, row in df.iterrows():
        ch = str(row[ch_col]).strip()
        if not ch or len(ch) != 1:
            continue
        bucket = {}
        for d in dialect_cols:
            val = str(row[d]).strip()
            if val.lower() == "nan" or val == "-":
                val = ""
            if val:
                bucket[d] = unicodedata.normalize("NFC", val)
        if bucket:
            data[ch] = bucket
    print(f"[Combined] {len(data)} chars; {len(dialect_cols)} dialect columns.")
    return data, dialect_cols

# ===== Hanzi / Hangul regexes for routing =====
_RE_HANGUL = re.compile(r"[\u1100-\u11FF\u3130-\u318F\uAC00-\uD7AF]+")
_HANZI_RE = re.compile(r"[\u3400-\u9FFF\U00020000-\U0002FFFF]")

def _has_hangul(s: str) -> bool:
    return bool(_RE_HANGUL.search(s or ""))

def _looks_like_hanzi(text: str) -> bool:
    return bool(_HANZI_RE.search(text or ""))

def _split_opts(s):
    return [
        t
        for t in re.split(r"[\/\|,;]+\s*|\s+", (s or "").strip())
        if t
    ]

def _concat_per_char(dialect, s):
    if not s:
        return ""
    out = []
    for ch in s:
        bucket = state["data"].get(ch, {})
        val = bucket.get(dialect, "")
        tok = _split_opts(val)
        out.append(tok[0] if tok else ch)
    return " ".join(x for x in out if x)

def _sorted_cols(cols):
    return sorted(
        cols,
        key=lambda k: (
            PREFERRED_ORDER.index(k) if k in PREFERRED_ORDER else 10_000,
            k.lower(),
        ),
    )

# ===== ML-tag styling + collapsible locality / provenance helpers =====
def _render_readings_html(val: str) -> str:
    """Escape a wide-CSV cell, styling any [ML]-tagged variant as inferred."""
    toks = [t for t in re.split(r"\s*\|\s*|\s+", (val or "").strip()) if t]
    out = []
    for t in toks:
        if t.endswith("[ML]"):
            base = t[:-4]
            out.append(
                "<span style=\"color:#9aa0a6;font-style:italic\" "
                "title=\"Machine-predicted - not attested in any source\">"
                f"{_html.escape(base)}<sup>ML</sup></span>"
            )
        else:
            out.append(_html.escape(t))
    return " &nbsp;".join(out)

# ----- IPA -> rough English approximation (for non-IPA readers) -----
# Longest-match transliteration. Voiceless unaspirated stops are written as
# English voiced letters (b/d/g) and aspirated as p/t/k, matching how they
# strike an English ear; tone letters become superscript pitch digits (5=high).
_IPA_ENG = {
    "tsʰ": "ts", "tɕʰ": "ch", "tʂʰ": "chr", "tʰ": "t", "pʰ": "p", "kʰ": "k",
    "cʰ": "ch", "ʈʰ": "tr", "kʷ": "kw", "ɡʷ": "gw",
    "ts": "dz", "tɕ": "j", "tʂ": "jr", "dʑ": "j", "dz": "dz", "dʐ": "jr",
    "uai": "wy", "iau": "yow", "uei": "way", "iou": "yoh", "yɛ": "yweh",
    "ai": "igh", "ɑi": "igh", "ɐi": "igh", "ei": "ay", "au": "ow", "ɑu": "ow",
    "ɐu": "ow", "ou": "oh", "ɔu": "ow",
    "ui": "wee", "ua": "wah", "uo": "waw", "uɔ": "waw", "uɛ": "weh",
    "uᴇ": "weh", "ue": "way", "uə": "wuh",
    "ia": "yah", "iɛ": "yeh", "iᴇ": "yeh", "ie": "yeh", "io": "yoh",
    "iu": "yoo", "iə": "yuh", "yn": "ewn", "ym": "ewm",
    "ɲ": "ny", "ŋ": "ng", "ɳ": "n", "ɱ": "m",
    "p": "b", "t": "d", "k": "g", "c": "j", "ʈ": "dr", "ɖ": "dr", "b": "b",
    "d": "d", "ɡ": "g", "g": "g", "q": "g", "ɢ": "g",
    "ɕ": "sh", "ʂ": "sh", "ʃ": "sh", "ʑ": "zh", "ʐ": "r", "ʒ": "zh", "s": "s",
    "z": "z", "f": "f", "v": "v", "h": "h", "x": "h", "ɣ": "gh", "ɦ": "h", "χ": "h",
    "m": "m", "n": "n", "l": "l", "ɬ": "hl", "ɭ": "l", "ɾ": "r", "r": "r",
    "ɹ": "r", "j": "y", "ɥ": "yu", "w": "w", "ʋ": "v", "ʔ": "'",
    "i": "ee", "y": "ew", "ɨ": "i", "ʉ": "u", "ɯ": "oo", "u": "oo",
    "ɪ": "ih", "ʏ": "ew", "ʊ": "uu", "e": "ay", "ᴇ": "eh", "ø": "er", "ɘ": "uh",
    "ɵ": "er", "ɤ": "uh", "o": "oh", "ə": "uh", "ɛ": "eh", "œ": "er", "ɜ": "er",
    "ʌ": "uh", "ɔ": "aw", "æ": "a", "ɐ": "uh", "a": "ah", "ɑ": "ah", "ɒ": "aw",
    "ɿ": "rr", "ʅ": "rr", "ɚ": "er", "ɝ": "er",
    "ʰ": "h", "ː": "", "ˤ": "", "ʲ": "y", "ʷ": "w", "ⁿ": "n",
    "\u0303": "n", "\u0329": "",
    "˥": "\u2075", "˦": "\u2074", "˧": "\u00b3", "˨": "\u00b2", "˩": "\u00b9",
}
_IPA_KEYS = sorted(_IPA_ENG, key=len, reverse=True)

# Columns whose readings are IPA (and thus benefit from the approximation).
_IPA_APPROX_COLS = {
    "Hokkien", "Teochew", "Hakka", "Wu", "Gan", "Xiang", "Jin",
    "Eastern Min (Fuzhou)", "Northern Min (Jian\'ou)", "Hainanese (Haikou)",
    "Hui (Shexian)", "Middle Chinese (Baxter)", "Old Chinese (Zhengzhang)",
}

def ipa_to_english(reading: str) -> str:
    r = unicodedata.normalize("NFD", (reading or "").replace("[ML]", "").strip())
    out = []
    i, L = 0, len(r)
    while i < L:
        if r[i] == " ":
            out.append(" "); i += 1; continue
        for k in _IPA_KEYS:
            if r.startswith(k, i):
                out.append(_IPA_ENG[k]); i += len(k); break
        else:
            if not unicodedata.combining(r[i]):
                out.append(r[i] if r[i].isalpha() else "")
            i += 1
    return "".join(out).strip()

def _ipa_approx_html(val: str) -> str:
    aps = []
    for v in [t for t in re.split(r"\s*\|\s*", (val or "").strip()) if t]:
        a = ipa_to_english(v)
        if a and a not in aps:
            aps.append(a)
    if not aps:
        return ""
    return ("&nbsp;&nbsp;<span style='color:#7fae7f;font-size:90%' "
            "title='Rough English approximation of the IPA - not exact'>"
            "\u2248 " + _html.escape(" / ".join(aps)) + "</span>")

_DIAL_BRANCH_RE = re.compile(r"^dial:([^-]+)-(.+)$")

def _render_locality_section(ch, bucket, dial_cols):
    """Collapsible per-locality (dial:<branch>-<place>) readings, by branch."""
    present = [(c, bucket[c]) for c in dial_cols if c in bucket and bucket[c]]
    if not present:
        return
    groups = {}
    for col, val in present:
        m = _DIAL_BRANCH_RE.match(col)
        branch = m.group(1) if m else "Other"
        place = m.group(2) if m else col
        groups.setdefault(branch, []).append((place, val))
    rows = []
    for branch, items in groups.items():
        parts = []
        for p, v in items:
            cell = f"<b>{_html.escape(p)}</b> {_render_readings_html(v)}"
            ap = ipa_to_english(v)
            if ap:
                cell += f" <span style='color:#7fae7f;font-size:88%'>\u2248 {_html.escape(ap)}</span>"
            parts.append(cell)
        cells = " &nbsp; ".join(parts)
        rows.append(
            f"<div style='margin:2px 0'><i>{_html.escape(branch)}</i>: {cells}</div>"
        )
    body = widgets.HTML(
        "<div style='font-size:90%;max-height:300px;overflow:auto'>"
        + "".join(rows) + "</div>"
    )
    acc = widgets.Accordion(children=[body])
    acc.set_title(0, f"Locality readings ({len(present)} points)")
    acc.selected_index = None
    display(acc)

def _show_provenance(ch):
    rows = state.get("prov", {}).get(ch, [])
    if not rows:
        display(HTML("<i style='color:#888'>No provenance recorded.</i>"))
        return
    head = "<tr><th>Column</th><th>Reading</th><th>Source</th><th>Conf</th></tr>"
    body = []
    for col, reading, source, conf in rows:
        c = "" if (conf is None or conf == "" or (isinstance(conf, float) and conf != conf)) else f"{float(conf):.2f}"
        sty = " style='color:#9aa0a6;font-style:italic'" if source == "ml" else ""
        body.append(
            f"<tr{sty}><td>{_html.escape(str(col))}</td>"
            f"<td>{_html.escape(str(reading))}</td>"
            f"<td>{_html.escape(str(source))}</td><td>{c}</td></tr>"
        )
    display(HTML(
        "<div style='max-height:360px;overflow:auto;border:1px solid #444;"
        "border-radius:8px;padding:6px'>"
        "<table style='width:100%;font-size:88%;border-collapse:collapse'>"
        + head + "".join(body) + "</table></div>"
    ))

def _render_provenance_toggle(ch):
    if not state.get("prov", {}).get(ch):
        return
    btn, out = make_toggle_list_button(
        "Show sources (provenance)", _show_provenance, ch, width=240
    )
    display(widgets.HBox([btn]))
    display(out)

def load_provenance(root):
    """char -> [(column, reading, source, conf), ...] from readings_long.parquet."""
    p = Path(root) / "data" / "out" / "readings_long.parquet"
    if not p.exists():
        return {}
    try:
        ldf = pd.read_parquet(p)
    except Exception:
        return {}
    keep = [c for c in ("char", "column", "reading", "source", "conf") if c in ldf.columns]
    prov = {}
    for r in ldf[keep].itertuples(index=False):
        ch = str(getattr(r, "char", "") or "")
        if len(ch) != 1:
            continue
        prov.setdefault(ch, []).append((
            getattr(r, "column", ""), getattr(r, "reading", ""),
            getattr(r, "source", ""), getattr(r, "conf", None),
        ))
    return prov

def ko_from_hanzi_local(src_hanzi: str):
    try:
        hangul = hanja_to(src_hanzi, "substitution") or ""
    except Exception:
        hangul = ""
    try:
        rom = _KO_TRANSLITER.translit(hangul) if hangul else ""
    except Exception:
        rom = ""
    return hangul, rom

def per_char_pairs_from_csv(text: str, dialect: str) -> str:
    pairs = []
    for ch in text:
        bucket = state["data"].get(ch, {})
        val = bucket.get(dialect, "")
        tok = _split_opts(val)
        pairs.append(f"{ch}({tok[0] if tok else ch})")
    return " ".join(pairs)

def ko_per_char_pairs(text: str) -> str:
    pairs = []
    for ch in text:
        try:
            hg = hanja_to(ch, "substitution") or ""
        except Exception:
            hg = ""
        try:
            rom = _KO_TRANSLITER.translit(hg) if hg else ""
        except Exception:
            rom = ""
        if hg and rom:
            pairs.append(f"{ch}({hg} {rom})")
        elif hg:
            pairs.append(f"{ch}({hg})")
        else:
            pairs.append(ch)
    return " ".join(pairs)

# ===== UI widgets (unified input) =====
main_input = widgets.Text(
    placeholder="Enter Hanzi (single/multi) or English",
    layout=widgets.Layout(width="340px"),
)
go_btn = widgets.Button(description="Go", button_style="success")
legend_btn = widgets.Button(description="IPA Legend")
readme_btn = widgets.Button(description="READ ME", button_style="info")
sinitic_btn = widgets.Button(
    description="Sinitic Evolution Paper", button_style="warning"
)

out_area = widgets.Output(
    layout={"border": "1px solid #333", "max_height": "560px", "overflow": "auto"}
)
legend_out = widgets.Output(
    layout={"border": "1px solid #333", "max_height": "220px", "overflow": "auto"}
)
readme_out = widgets.Output(
    layout={"border": "1px solid #333", "max_height": "800px", "overflow": "auto"}
)
sinitic_out = widgets.Output(
    layout={"border": "1px solid #333", "max_height": "800px", "overflow": "auto"}
)
reverse_out = widgets.Output(
    layout={"border": "1px solid #333", "max_height": "800px", "overflow": "auto"}
)

state = {"data": None, "dialect_cols": [], "symbols": {}, "prov": {}, "preload_msg": ""}

# ---- Auto-load on run ----
try:
    state["data"], state["dialect_cols"] = load_from_combined_csv(COMBINED_CSV)
    state["symbols"] = load_phonetic_symbols(IPA_FILE)
    state["prov"] = load_provenance(ROOT)
    state["preload_msg"] = "✅ Auto-loaded dataset. Ready."
except Exception as e:
    state["preload_msg"] = f"❌ Auto-load failed: {e}"

# ===== Main lookup (Hanzi) =====
def on_lookup(_):
    with out_area:
        clear_output()
        if state["data"] is None:
            print("Auto-load failed. Fix paths and re-run.")
            return

        s = (main_input.value or "").strip()
        if not s:
            print("Enter one or more Hanzi.")
            return

        # ---------- SINGLE CHARACTER ----------
        if len(s) == 1:
            ch = s
            bucket = state["data"].get(ch, {})
            if not bucket:
                print(f"No data for '{ch}'")
                return

            display(
                HTML(f"<b>Pronunciations for '{_html.escape(ch)}':</b>")
            )
            _dial_cols = [d for d in state["dialect_cols"] if d.startswith("dial:")]
            for dialect in _sorted_cols(state["dialect_cols"]):
                if dialect.startswith("dial:"):
                    continue
                if dialect not in bucket:
                    continue
                val = bucket[dialect]
                dl = _norm(dialect)

                display(HTML(f"<b>{_html.escape(dialect)}:</b>"))

                if dialect == "Japanese":
                    for tok in [t for t in val.split() if t]:
                        if any(
                            "\u3040" <= c <= "\u30FF"
                            or "\u31F0" <= c <= "\u31FF"
                            for c in tok
                        ):
                            _make_button_and_play(
                                tok,
                                *VOICE_JA,
                                speak=_prep_ja_tts(tok, tok),
                            )
                        else:
                            display(
                                HTML(
                                    f"&nbsp;&nbsp;<code>{_html.escape(tok)}</code>"
                                )
                            )
                    continue

                if dialect in ("Japanese On", "Japanese Kun"):
                    display(
                        HTML(
                            f"&nbsp;&nbsp;<code>{_html.escape(_inline_tokens(val))}</code>"
                        )
                    )
                    continue

                if dialect == "Korean (Hangul)":
                    runs = _RE_HANGUL.findall(val) or []
                    if runs:
                        for run in runs:
                            _make_button_and_play(run, *VOICE_KO)
                    else:
                        display(
                            HTML(
                                f"&nbsp;&nbsp;<code>{_html.escape(val)}</code>"
                            )
                        )
                    continue

                if dialect == "Korean (romanized)":
                    display(
                        HTML(
                            f"&nbsp;&nbsp;<code>{_html.escape(_inline_tokens(val))}</code>"
                        )
                    )
                    continue

                if "vietnam" in dl:
                    line = _inline_tokens(val)
                    display(
                        HTML(
                            f"&nbsp;&nbsp;<code>{_html.escape(line)}</code>"
                        )
                    )
                    continue

                if dl.startswith("mandarin"):
                    display(
                        HTML(
                            f"&nbsp;&nbsp;<code>{_html.escape(_inline_tokens(val))}</code>"
                        )
                    )
                    _make_button_and_play(ch, *VOICE_ZH)
                    continue

                if dl.startswith("cantonese"):
                    display(
                        HTML(
                            f"&nbsp;&nbsp;<code>{_render_readings_html(val)}</code>"
                        )
                    )
                    _make_button_and_play(ch, *VOICE_YUE)
                    continue

                if (
                    ENABLE_TOPOLECT_FALLBACK_CHAR_TTS
                    and dialect in TOPOLECT_LABELS
                ):
                    display(
                        HTML(
                            f"&nbsp;&nbsp;<code>{_html.escape(val)}</code>"
                        )
                    )
                    _make_button_and_play(ch, *TOPOLECT_FALLBACK_VOICE)
                    continue

                display(
                    HTML(
                        f"&nbsp;&nbsp;<code>{_render_readings_html(val)}</code>"
                    )
                )
                if dialect in _IPA_APPROX_COLS:
                    _ap = _ipa_approx_html(val)
                    if _ap:
                        display(HTML(_ap))

            _render_locality_section(ch, bucket, _dial_cols)
            _render_provenance_toggle(ch)
            return

        # ---------- MULTI-CHARACTER ----------
        text = s
        display(
            HTML(f"<b>Pronunciations for '{_html.escape(text)}':</b>")
        )

        # Definition block — solely Google Translate
        display(HTML("<b>Definition:</b>"))
        en_common = gtrans_translate_text(text, target="en", source="zh-Hant")

        zh_mid = gtrans_translate_text(text, target="zh-TW") or text
        ja_mid = gtrans_translate_text(text, target="ja") or text
        ko_mid = gtrans_translate_text(text, target="ko") or text
        vi_mid = gtrans_translate_text(text, target="vi") or text

        if zh_mid == text and en_common:
            zh_try = gtrans_translate_text(en_common, target="zh-TW", source="en")
            if zh_try:
                zh_mid = zh_try

        if (not _has_hangul(ko_mid)) and en_common:
            ko_try = gtrans_translate_text(en_common, target="ko", source="en")
            if ko_try and _has_hangul(ko_try):
                ko_mid = ko_try

        zh_rom = _rom_zh(zh_mid)
        ja_rom_def = gtrans_romanize(ja_mid, "ja") or ""
        ko_rom_def = _rom_ko(ko_mid) if _has_hangul(ko_mid) else ""

        display(
            HTML(
                f"&nbsp;&nbsp;<code>English (common): {en_common or '(n/a)'}</code>"
            )
        )
        display(
            HTML(
                f"&nbsp;&nbsp;<code>Chinese (zh): {zh_mid}{(' — ' + zh_rom) if zh_rom else ''}</code>"
            )
        )
        display(
            HTML(
                f"&nbsp;&nbsp;<code>Japanese (ja): {ja_mid}{(' — ' + ja_rom_def) if ja_rom_def else ''}</code>"
            )
        )
        display(
            HTML(
                f"&nbsp;&nbsp;<code>Korean (ko): {ko_mid}{(' — ' + ko_rom_def) if ko_rom_def else ''}</code>"
            )
        )
        display(
            HTML(f"&nbsp;&nbsp;<code>Vietnamese (vi): {vi_mid}</code>")
        )
        _maybe_diag()

        # Japanese TTS via kana
        display(HTML("<b>Japanese:</b>"))
        jp_shinjitai = to_shinjitai_per_char(text)
        ja_romaji, ja_kana = ja_romaji_and_kana(text)
        display(
            HTML(
                f"&nbsp;&nbsp;<code>{_html.escape(ja_romaji or '(n/a)')}</code>"
            )
        )
        _make_button_and_play(
            jp_shinjitai,
            *VOICE_JA,
            speak=_prep_ja_tts(ja_kana, jp_shinjitai),
        )

        # Korean
        display(HTML("<b>Korean (Hangul):</b>"))
        ko_hangul, ko_romaji = ko_from_hanzi_local(text)
        if ko_hangul:
            # Per-character Hanja info from koreanhanja.app/<泥棒>, etc.
            hanja_btn, hanja_out = make_toggle_list_button(
                "Hanja (per character via koreanhanja.app)",
                show_hanja_rows_from_hanzi,
                text,
            )
            display(widgets.HBox([hanja_btn]))
            display(hanja_out)
            _make_button_and_play(ko_hangul, *VOICE_KO)
        else:
            display(HTML("&nbsp;&nbsp;<code>(n/a)</code>"))

        display(HTML("<b>Korean (romanized):</b>"))
        display(
            HTML(
                f"&nbsp;&nbsp;<code>{_html.escape(ko_romaji or '(n/a)')}</code>"
            )
        )
        display(
            HTML(f"&nbsp;&nbsp;<code>{_html.escape(ko_per_char_pairs(text))}</code>")
        )

        # Vietnamese Nôm table from Hanzi
        display(HTML("<b>Vietnamese:</b>"))
        nom_btn, nom_out = make_toggle_list_button(
            " Chữ Nôm (with Quốc Ngữ audio)", show_nom_table_from_hanzi, text
        )
        display(widgets.HBox([nom_btn]))
        display(nom_out)

        # Other dialects (skip Viet, JP, KO)
        for dialect in _sorted_cols(state["dialect_cols"]):
            dl = _norm(dialect)
            if dialect.startswith("dial:"):
                continue
            if dl in ("definition", "fanqie", "fanqie_upper", "fanqie_lower"):
                continue
            if dl in ("japanese", "japanese on", "japanese kun"):
                continue
            if dl.startswith("korean"):
                continue
            if "vietnam" in dl:
                continue

            if dl.startswith("mandarin"):
                line = _inline_tokens(_concat_per_char(dialect, text))
                if line:
                    display(HTML(f"<b>{_html.escape(dialect)}:</b>"))
                    display(
                        HTML(f"&nbsp;&nbsp;<code>{_html.escape(line)}</code>")
                    )
                    display(
                        HTML(
                            f"&nbsp;&nbsp;<code>{_html.escape(per_char_pairs_from_csv(text, dialect))}</code>"
                        )
                    )
                    _make_button_and_play(text, *VOICE_ZH)
                continue

            if dl.startswith("cantonese"):
                line = _inline_tokens(_concat_per_char(dialect, text))
                if line:
                    display(HTML(f"<b>{_html.escape(dialect)}:</b>"))
                    display(
                        HTML(f"&nbsp;&nbsp;<code>{_html.escape(line)}</code>")
                    )
                    display(
                        HTML(
                            f"&nbsp;&nbsp;<code>{_html.escape(per_char_pairs_from_csv(text, dialect))}</code>"
                        )
                    )
                    _make_button_and_play(text, *VOICE_YUE)
                continue

            if ENABLE_TOPOLECT_FALLBACK_CHAR_TTS and dialect in TOPOLECT_LABELS:
                line = _inline_tokens(_concat_per_char(dialect, text))
                if line:
                    display(HTML(f"<b>{_html.escape(dialect)}:</b>"))
                    display(
                        HTML(
                            f"&nbsp;&nbsp;<code>{_html.escape(line)}</code>"
                        )
                    )
                    _make_button_and_play(text, *TOPOLECT_FALLBACK_VOICE)
                continue

            line = _inline_tokens(_concat_per_char(dialect, text))
            if line:
                display(HTML(f"<b>{_html.escape(dialect)}:</b>"))
                display(
                    HTML(f"&nbsp;&nbsp;<code>{_html.escape(line)}</code>")
                )

# ===== IPA legend =====
def on_legend(_):
    with legend_out:
        clear_output()
        if state["data"] is None:
            print("Auto-load failed. Fix paths and re-run.")
            return
        s = (main_input.value or "").strip()
        if not s:
            print("Enter a loaded Hanzi first.")
            return
        if len(s) == 1 and s in state["data"]:
            readings = [v for v in state["data"][s].values() if v]
        else:
            readings = [
                _concat_per_char(d, s) for d in state["dialect_cols"]
            ]
        lines = show_ipa_legend_lines(readings, state["symbols"])
        if not lines:
            print("No IPA symbols found (or IPA file missing).")
            return
        display(HTML("<b>📘 IPA Legend:</b>"))
        for ln in lines:
            display(HTML(f"<code>{_html.escape(ln)}</code>"))

# ===== Guides / README =====
def on_readme(_):
    with readme_out:
        clear_output()
        display(HTML("<b>📚 Guides (README)</b>"))

        guide_dropdown = widgets.Dropdown(
            options=list(GUIDE_FILES.keys()),
            description="Guide:",
            layout=widgets.Layout(width="320px"),
        )

        open_btn = widgets.Button(
            description="Open",
            button_style="info",
            layout=widgets.Layout(width="100px"),
        )

        guide_content_out = widgets.Output(
            layout={"border": "1px solid #333", "overflow": "auto"}
        )

        def _open(_b):
            with guide_content_out:
                clear_output()
                title = guide_dropdown.value
                fname = GUIDE_FILES.get(title)
                if not fname:
                    display(
                        HTML("<i style='color:#c66'>Unknown guide key.</i>")
                    )
                    return
                path = GUIDES_DIR / fname
                if not path.exists():
                    display(
                        HTML(
                            f"<i style='color:#c66'>Missing: {path}</i>"
                        )
                    )
                    return
                try:
                    text = path.read_text(encoding="utf-8")
                except Exception as e:
                    display(
                        HTML(
                            f"<i style='color:#c66'>Error reading {path}: {e}</i>"
                        )
                    )
                    return

                display(Markdown(f"## {title}\n\n{text}"))

        open_btn.on_click(_open)
        controls = widgets.HBox([guide_dropdown, open_btn])
        display(widgets.VBox([controls, guide_content_out]))

# ===== Sinitic evolution paper =====
def on_sinitic_open(_):
    with sinitic_out:
        clear_output()
        paper_path = (
            GUIDES_DIR
            / "Historical and Phonological Evolution of Sinitic (Chinese) Languages.docx"
        )
        if not paper_path.exists():
            display(
                HTML("<i style='color:#c66'>Paper file not found.</i>")
            )
            return
        try:
            doc = Document(str(paper_path))
            text = "\n\n".join(
                p.text for p in doc.paragraphs if p.text.strip()
            )
            display(
                Markdown(
                    "## Historical and Phonological Evolution of Sinitic (Chinese) Languages\n\n"
                    + text
                )
            )
        except Exception as e:
            display(
                HTML(
                    f"<i style='color:#c66'>Error reading DOCX: {e}</i>"
                )
            )

# ===== Reverse lookup from English =====
def on_reverse_lookup(_):
    with reverse_out:
        clear_output()
    with out_area:
        clear_output()

    if state["data"] is None:
        with reverse_out:
            print("Auto-load failed. Fix paths and re-run.")
        return

    en = (main_input.value or "").strip()
    with reverse_out:
        if not en:
            print("Enter an English word or phrase.")
            return

        display(
            HTML(
                f"<h4>Reverse lookup for: <code>{_html.escape(en)}</code></h4>"
            )
        )

        # --- 1) Chinese dialects via EN → zh-TW ---
        display(
            HTML("<b>Chinese dialects (via Traditional Chinese base phrase):</b>")
        )
        if not _GT_READY:
            display(
                HTML(
                    "<i style='color:#c66'>Translation client not ready; cannot perform EN→ZH pipeline.</i>"
                )
            )
        else:
            zh = gtrans_translate_text(en, target="zh-TW", source="en")
            if not zh:
                display(
                    HTML(
                        "<i style='color:#c66'>Translation to Traditional Chinese failed.</i>"
                    )
                )
            else:
                display(
                    HTML(f"&nbsp;&nbsp;<code>{_html.escape(zh)}</code>")
                )
                for dialect in _sorted_cols(state["dialect_cols"]):
                    dl = _norm(dialect)
                    if dialect.startswith("dial:"):
                        continue
                    if dl in (
                        "definition",
                        "fanqie",
                        "fanqie_upper",
                        "fanqie_lower",
                    ):
                        continue
                    if "japanese" in dl or dl.startswith("korean") or "vietnam" in dl:
                        continue
                    line = _inline_tokens(_concat_per_char(dialect, zh))
                    if line:
                        display(
                            HTML(f"<b>{_html.escape(dialect)}:</b>")
                        )
                        display(
                            HTML(
                                f"&nbsp;&nbsp;<code>{_html.escape(line)}</code>"
                            )
                        )
                        display(
                            HTML(
                                f"&nbsp;&nbsp;<code>{_html.escape(per_char_pairs_from_csv(zh, dialect))}</code>"
                            )
                        )

        # --- 2) Vietnamese via EN → vi → Nom (Quốc Ngữ mode) ---
        display(
            HTML(
                "<hr><b>Vietnamese via Nom Foundation (Quốc Ngữ, input_type=rqn_or_hn):</b>"
            )
        )
        if not _GT_READY:
            display(
                HTML(
                    "<i style='color:#c66'>Translation client not ready; skipping Vietnamese step.</i>"
                )
            )
        else:
            vi = gtrans_translate_text(en, target="vi", source="en")
            if not vi:
                display(
                    HTML(
                        "<i style='color:#c66'>Translation to Vietnamese failed.</i>"
                    )
                )
            else:
                display(
                    HTML(f"&nbsp;&nbsp;<code>{_html.escape(vi)}</code>")
                )

                df, url = fetch_nom_dict_table_from_vi_or_en(vi)
                if df is None or df.empty:
                    display(
                        HTML("<i style='color:#888'>No Nôm dictionary results.</i>")
                    )
                else:
                    render_nom_table_with_vi_tts(df)

                display(
                    HTML(f'<a href="{url}" target="_blank">View full page</a>')
                )

        # --- 3) Korean via koreanhanja.app English query ---
        display(HTML("<hr><b>Korean via koreanhanja.app (English query):</b>"))
        try:
            rows_ko, url_ko = fetch_hanja_rows_from_english(en)
            render_koreanhanja_table(rows_ko, url_ko)
        except Exception as e:
            display(
                HTML(
                    f"<i style='color:#c66'>Error contacting koreanhanja.app: {e}</i>"
                )
            )

        # --- 4) Japanese via JapanDict English search (multi-page) ---
        display(
            HTML("<hr><b>Japanese via JapanDict (English search):</b>")
        )
        try:
            entries_jp, url_jp = fetch_japandict_from_english(en)
            render_japandict_entries(entries_jp, url_jp)
        except Exception as e:
            display(
                HTML(
                    f"<i style='color:#c66'>Error contacting JapanDict: {e}</i>"
                )
            )

# ===== Unified router =====
def on_go(_):
    txt = (main_input.value or "").strip()
    if not txt:
        with out_area:
            clear_output()
            print("Enter Hanzi or English.")
        with reverse_out:
            clear_output()
        return

    if _looks_like_hanzi(txt):
        on_lookup(None)
    else:
        on_reverse_lookup(None)

# Wire UI
go_btn.on_click(on_go)
legend_btn.on_click(on_legend)
readme_btn.on_click(on_readme)
sinitic_btn.on_click(on_sinitic_open)

# Layout + preload status
out_area.append_display_data(state["preload_msg"])
display(
    widgets.VBox(
        [
            widgets.HBox(
                [main_input, go_btn, legend_btn, readme_btn, sinitic_btn]
            ),
            out_area,
            legend_out,
            readme_out,
            sinitic_out,
            widgets.HTML("<hr>"),
            reverse_out,
        ]
    )
)


Loaded 369 variant mappings (jp).
[Combined] 67552 chars; 97 dialect columns.
